In [1]:
import pandas as pd
import numpy as np
import requests

#### ISIN Codes mapping

In [2]:
df_isin_map = pd.read_excel(r"D:\FY25-26\Quant Fund\data\Investment Team Data\ISIN Ticker Mapping.xlsx",sheet_name='Sheet1')

In [3]:
df_isin_map.head(5)

,Stocks,Bloomberg Code,ISIN,Name
0,NIFTY Midcap 100,NSEMCAP Index,NaN,NaN
1,NaN,RELIANCE IN Equity,INE002A01018,RELIANCE INDUSTRIES LIMITED
2,NaN,TCS IN Equity,INE467B01029,TATA CONSULTANCY SERVICES LIMITED
3,NaN,HDFCB IN Equity,INE040A01034,HDFC BANK LIMITED
4,NaN,BHARTI IN Equity,INE397D01024,Bharti Airtel Limited


In [4]:
df_ticker_map = pd.read_excel(r"D:\FY25-26\Quant Fund\data\Investment Team Data\ISIN Ticker Mapping.xlsx",sheet_name='Sheet2')

In [5]:
df_ticker_map.head()

,Ticker,Company Name,Ticker.1
0,NSEMCAP,NIFTY Midcap 100,NSEMCAP
1,RELIANCE IN,Reliance Industries Ltd,RELIANCE IN
2,TCS IN,Tata Consultancy Services Ltd,TCS IN
3,HDFCB IN,HDFC Bank Ltd,HDFCB IN
4,BHARTI IN,Bharti Airtel Ltd,BHARTI IN


In [6]:
import difflib

In [7]:
df_isin_map['Name']

0                                       NaN
1               RELIANCE INDUSTRIES LIMITED
2         TATA CONSULTANCY SERVICES LIMITED
3                         HDFC BANK LIMITED
4                     Bharti Airtel Limited
                       ...                 
1039               YASHO INDUSTRIES LIMITED
1040               GOCL CORPORATION LIMITED
1041    MADHYA BHARAT AGRO PRODUCTS LIMITED
1042             Jubilant FoodWorks Limited
1043                BRIGHTCOM GROUP LIMITED
Name: Name, Length: 1044, dtype: object

In [8]:
difflib.get_close_matches(word="Reliance Industries Ltd".lower(),possibilities=["RELIANCE INDUSTRIES LIMITED".lower()],cutoff=0.8)

['reliance industries limited']

In [9]:
df_isin_map

,Stocks,Bloomberg Code,ISIN,Name
0,NIFTY Midcap 100,NSEMCAP Index,NaN,NaN
1,NaN,RELIANCE IN Equity,INE002A01018,RELIANCE INDUSTRIES LIMITED
2,NaN,TCS IN Equity,INE467B01029,TATA CONSULTANCY SERVICES LIMITED
3,NaN,HDFCB IN Equity,INE040A01034,HDFC BANK LIMITED
4,NaN,BHARTI IN Equity,INE397D01024,Bharti Airtel Limited
...,...,...,...,...
1039,NaN,YASHO INDUSTRIES LIMITED,INE616Z01012,YASHO INDUSTRIES LIMITED
1040,NaN,GOCL IN Equity,INE077F01035,GOCL CORPORATION LIMITED
1041,NaN,NaN,INE900L01010,MADHYA BHARAT AGRO PRODUCTS LIMITED
1042,NaN,JUBI IN Equity,INE797F01020,Jubilant FoodWorks Limited


In [10]:
def get_best_match(ticker, possibilities, cutoff=0.8):
    matches = difflib.get_close_matches(word=str(ticker).lower(), possibilities=[str(p).lower() for p in possibilities], cutoff=cutoff)
    return matches[0] if matches else None

# Create a dictionary to map tickers from df1 to df2 using fuzzy matching
ticker_mapping = {ticker: get_best_match(ticker, df_ticker_map['Company Name']) for ticker in df_isin_map['Name']}


In [11]:
pd.DataFrame(ticker_mapping.items())

,0,1
0,NaN,None
1,RELIANCE INDUSTRIES LIMITED,reliance industries ltd
2,TATA CONSULTANCY SERVICES LIMITED,tata consultancy services ltd
3,HDFC BANK LIMITED,hdfc bank ltd
4,Bharti Airtel Limited,bharti airtel ltd
...,...,...
995,YASHO INDUSTRIES LIMITED,yasho industries ltd
996,GOCL CORPORATION LIMITED,None
997,MADHYA BHARAT AGRO PRODUCTS LIMITED,None
998,Jubilant FoodWorks Limited,jubilant foodworks ltd


In [12]:
df_isin_map['Name'].apply(lambda x: difflib.get_close_matches(str(x).lower(), df_ticker_map['Company Name'].astype(str).apply(lambda x:x.lower())))

0                                                      []
1       [reliance industries ltd, rain industries ltd,...
2       [tata consultancy services ltd, max financial ...
3             [hdfc bank ltd, dcb bank ltd, csb bank ltd]
4                 [bharti airtel ltd, bharti hexacom ltd]
                              ...                        
1039    [yasho industries ltd, alok industries ltd, pr...
1040    [gocl corp ltd, navkar corporation ltd, genesy...
1041    [hatsun agro product ltd, astra microwave prod...
1042    [jubilant foodworks ltd, jubilant industries l...
1043                                                   []
Name: Name, Length: 1044, dtype: object

In [13]:
# from fuzzywuzzy import fuzz
from fuzzywuzzy import process

d:\.conda\envs\qf_3\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [14]:
def fuzzy_merge(df_1, df_2, key1, key2, threshold=90, limit=1):
    """
    :param df_1: the left table to join
    :param df_2: the right table to join
    :param key1: key column of the left table
    :param key2: key column of the right table
    :param threshold: how close the matches should be to return a match, based on Levenshtein distance
    :param limit: the amount of matches that will get returned, these are sorted high to low
    :return: dataframe with boths keys and matches
    """
    s = df_2[key2].tolist()
    
    m = df_1[key1].apply(lambda x: process.extract(x, s, limit=limit))    
    df_1['matches'] = m
    
    m2 = df_1['matches'].apply(lambda x: ', '.join([i[0] for i in x if i[1] >= threshold]))
    df_1['matches'] = m2
    
    return df_1

In [15]:
df_isin_map['Name'] = df_isin_map['Name'].astype(str)
df_ticker_map['Company Name'] = df_ticker_map['Company Name'].astype(str)

In [16]:
df_match = fuzzy_merge(df_1 = df_isin_map,df_2 = df_ticker_map, key1='Name',key2= 'Company Name', threshold=80)

In [22]:
df_match.head()

,Stocks,Bloomberg Code,ISIN,Name,matches
0,NIFTY Midcap 100,NSEMCAP Index,NaN,nan,Bajaj Finance Ltd
1,NaN,RELIANCE IN Equity,INE002A01018,RELIANCE INDUSTRIES LIMITED,Reliance Industries Ltd
2,NaN,TCS IN Equity,INE467B01029,TATA CONSULTANCY SERVICES LIMITED,Tata Consultancy Services Ltd
3,NaN,HDFCB IN Equity,INE040A01034,HDFC BANK LIMITED,HDFC Bank Ltd
4,NaN,BHARTI IN Equity,INE397D01024,Bharti Airtel Limited,Bharti Airtel Ltd


In [ ]:
df_match[df_match['ISIN'].isna()]

,Stocks,Bloomberg Code,ISIN,Name,matches
0,NIFTY Midcap 100,NSEMCAP Index,NaN,nan,Bajaj Finance Ltd
166,NaN,UTNIFTY IN Equity,NaN,UTI Nifty 50 ETF,UTI Nifty 50 ETF
376,NaN,NHIT IN Equity,NaN,NATIONAL HIGHWAYS INFRA TRUST UNT,National Highways Infra Trust
391,NaN,CPBI IN Equity,NaN,CENTURY PLYBOARDS (INDIA) LTD.,Century Plyboards India Ltd
433,NaN,RAINBOW IN Equity,NaN,RAINBOW CHILDREN'S MEDICARE LIMITED,Rainbow Children's Medicare Ltd
437,NaN,CUBEINVI IN Equity,NaN,CUBE HIGHWAYS UNT,National Highways Infra Trust
440,Bharat Bond ETF - April 2030,BETF430 IN Equity,NaN,nan,Bajaj Finance Ltd
505,NaN,GBEES IN Equity,NaN,Nippon India ETF Gold BeES,NIPPON INDIA ETF GOLD BEES
508,NaN,KKF IN Equity,NaN,KIRLOSKAR FERROUS INDUSTRIES LTD,Kirloskar Ferrous Industries Ltd
512,NaN,ARVINDPO IN Equity,NaN,ARVIND PORT AND INFRA LIMITED,Arvind Ltd


In [24]:
df_iter = df_match[df_match['Bloomberg Code'].isna()][['ISIN','Name']]

In [135]:
df_mapping_present = df_match[~df_match['Bloomberg Code'].isna()][['ISIN','Name','Bloomberg Code']]

In [25]:
df_match[df_match['matches']==''][df_match[df_match['matches']=='']['Bloomberg Code'].isna()]

,Stocks,Bloomberg Code,ISIN,Name,matches
187,NaN,NaN,INE699H01024,AWL AGRI BUSINESS LIMITED,
848,NaN,NaN,INE484C01030,VEEDOL CORPORATION LIMITED,


In [ ]:
df_iter.reset_index(drop=True,inplace=True)

In [62]:
import requests
import json
import time

In [204]:
missing_isin = ['INE663F01024', 'IN9047A01029', 'INE342G01023', 'IN9439E01012',
       'IN9671H01013']

In [205]:
figi_api_key = __import__('os').environ.get('OPENFIGI_API_KEY', '')  # set OPENFIGI_API_KEY in your environment
url = 'https://api.openfigi.com/v3/mapping'
headers = {'Content-Type':'text/json', 'X-OPENFIGI-APIKEY':figi_api_key }


payload_dict = [{"idType":"ID_ISIN","idValue":missing_isin[i],"exchCode":"IN"} for i in range(0,len(missing_isin))]
    # df_iter['Response'] = r
    # time.sleep(3)
# payload = json.dumps(payload_dict)


In [206]:
payload = json.dumps(payload_dict)

In [207]:
r = requests.post(url, headers=headers, data=payload)

In [208]:
r.json()

[{'warning': 'No identifier found.'},
 {'data': [{'figi': 'BBG01KXM82F0',
    'name': 'GRASIM INDUSTRIE-PARTLY PAID',
    'ticker': 'GRASIMPP',
    'exchCode': 'IN',
    'compositeFIGI': 'BBG01KXM82F0',
    'securityType': 'Common Stock',
    'marketSector': 'Equity',
    'shareClassFIGI': 'BBG01KXM82L3',
    'securityType2': 'Common Stock',
    'securityDescription': 'GRASIMPP'}]},
 {'data': [{'figi': 'BBG00KHSDKF0',
    'name': 'NIIT LEARNING SYSTEMS LTD',
    'ticker': 'NIITMTS',
    'exchCode': 'IN',
    'compositeFIGI': 'BBG00KHSDKF0',
    'securityType': 'Common Stock',
    'marketSector': 'Equity',
    'shareClassFIGI': 'BBG00KHSDKG9',
    'securityType2': 'Common Stock',
    'securityDescription': 'NIITMTS'}]},
 {'data': [{'figi': 'BBG01KXM4834',
    'name': 'SKIPPER LTD-PARTLY PAID',
    'ticker': 'SKIPPEPP',
    'exchCode': 'IN',
    'compositeFIGI': 'BBG01KXM4834',
    'securityType': 'Common Stock',
    'marketSector': 'Equity',
    'shareClassFIGI': 'BBG01KXM4889',
    'se

In [209]:
len(payload_dict)

5

In [95]:
payload = json.dumps(payload_dict[100:])
r2 = requests.post(url, headers=headers, data=payload)

In [210]:
r1_json = r.json()
r2_json = r2.json()

In [116]:
payload_dict[0]

{'idType': 'ID_ISIN', 'idValue': 'IN9047A01029', 'exchCode': 'IN'}

In [117]:
df_iter[df_iter['ISIN']=='IN9047A01029']

,ISIN,Name
0,IN9047A01029,GRASIM INDUSTRIES LTD


In [211]:
l1 = []
for i in range(0,len(r1_json)):
    print(i)
    print(r1_json[i])
    if 'warning' not in r1_json[i]:
        ticker = r1_json[i]['data'][0]['ticker']
    else:
        ticker = np.nan
    l1.append(ticker)


0
{'warning': 'No identifier found.'}
1
{'data': [{'figi': 'BBG01KXM82F0', 'name': 'GRASIM INDUSTRIE-PARTLY PAID', 'ticker': 'GRASIMPP', 'exchCode': 'IN', 'compositeFIGI': 'BBG01KXM82F0', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG01KXM82L3', 'securityType2': 'Common Stock', 'securityDescription': 'GRASIMPP'}]}
2
{'data': [{'figi': 'BBG00KHSDKF0', 'name': 'NIIT LEARNING SYSTEMS LTD', 'ticker': 'NIITMTS', 'exchCode': 'IN', 'compositeFIGI': 'BBG00KHSDKF0', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG00KHSDKG9', 'securityType2': 'Common Stock', 'securityDescription': 'NIITMTS'}]}
3
{'data': [{'figi': 'BBG01KXM4834', 'name': 'SKIPPER LTD-PARTLY PAID', 'ticker': 'SKIPPEPP', 'exchCode': 'IN', 'compositeFIGI': 'BBG01KXM4834', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG01KXM4889', 'securityType2': 'Common Stock', 'securityDescription': 'SKIPPEPP'}]}
4
{'data': [{'figi': 'BBG01N8ZLPR4',

In [212]:
l1

[nan, 'GRASIMPP', 'NIITMTS', 'SKIPPEPP', 'SOBHAPP']

In [126]:
l1 = []
for i in range(0,len(r1_json)):
    print(i)
    print(r1_json[i])
    if 'warning' not in r1_json[i]:
        ticker = r1_json[i]['data'][0]['ticker']
    else:
        ticker = np.nan
    l1.append(ticker)

l2 = []
for i in range(0,len(r2_json)):
    print(i)
    print(r2_json[i])
    if 'warning' not in r2_json[i]:
        ticker = r2_json[i]['data'][0]['ticker']
    else:
        ticker = np.nan
    l2.append(ticker)

0
{'data': [{'figi': 'BBG01KXM82F0', 'name': 'GRASIM INDUSTRIE-PARTLY PAID', 'ticker': 'GRASIMPP', 'exchCode': 'IN', 'compositeFIGI': 'BBG01KXM82F0', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG01KXM82L3', 'securityType2': 'Common Stock', 'securityDescription': 'GRASIMPP'}]}
1
{'data': [{'figi': 'BBG003DKJ5C9', 'name': 'ETERNAL LTD', 'ticker': 'ETERNAL', 'exchCode': 'IN', 'compositeFIGI': 'BBG003DKJ5C9', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG003DKJ5D8', 'securityType2': 'Common Stock', 'securityDescription': 'ETERNAL'}]}
2
{'data': [{'figi': 'BBG00GPZ5771', 'name': 'SBI LIFE INSURANCE CO LTD', 'ticker': 'SBILIFE', 'exchCode': 'IN', 'compositeFIGI': 'BBG00GPZ5771', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG00GPZ57D4', 'securityType2': 'Common Stock', 'securityDescription': 'SBILIFE'}]}
3
{'data': [{'figi': 'BBG000D0ZK56', 'name': 'TATA POWER CO LTD', 'ticker': 'TPWR', 'exc

In [128]:
pd.Series(l1).isna().any()

np.True_

In [132]:
df_iter['Ticker'] = pd.concat([pd.Series(l1),pd.Series(l2)],axis=0).reset_index(drop=True)

In [133]:
df_iter[df_iter['Ticker'].isna()]

,ISIN,Name,Ticker
57,INE725A01022,NAVA LIMITED,NaN
70,INE602W01019,Senco Gold Ltd,NaN
91,INE989A01024,SANGHVI MOVERS LIMITED,NaN


In [137]:
df_mapping_present.rename(columns={'Bloomberg Code':'Ticker'},inplace=True)

In [139]:
df_map_fin1 = pd.concat([df_mapping_present,df_iter],axis=0)

In [140]:
df_map_fin1.reset_index(drop=True,inplace=True)

In [142]:
df_map_fin1[df_map_fin1['Ticker'].isna()]

,ISIN,Name,Ticker
942,INE725A01022,NAVA LIMITED,NaN
955,INE602W01019,Senco Gold Ltd,NaN
976,INE989A01024,SANGHVI MOVERS LIMITED,NaN


In [202]:
# df_map_fin1.to_excel(r"D:\FY25-26\Quant Fund\data\ISIN_Ticker_Mapping_openfig.xlsx",index=False)

In [173]:
df_map_fin1[df_map_fin1['ISIN'].isna()]

,ISIN,Name,Ticker
0,NaN,nan,NSEMCAP Index
154,NaN,UTI Nifty 50 ETF,UTNIFTY IN Equity
341,NaN,NATIONAL HIGHWAYS INFRA TRUST UNT,NHIT IN Equity
354,NaN,CENTURY PLYBOARDS (INDIA) LTD.,CPBI IN Equity
392,NaN,RAINBOW CHILDREN'S MEDICARE LIMITED,RAINBOW IN Equity
395,NaN,CUBE HIGHWAYS UNT,CUBEINVI IN Equity
398,NaN,nan,BETF430 IN Equity
451,NaN,Nippon India ETF Gold BeES,GBEES IN Equity
454,NaN,KIRLOSKAR FERROUS INDUSTRIES LTD,KKF IN Equity
458,NaN,ARVIND PORT AND INFRA LIMITED,ARVINDPO IN Equity


In [147]:
isin_missing = df_map_fin1[df_map_fin1['ISIN'].isna()]['Ticker'].astype(str).str.replace(" IN Equity","").to_list()

In [148]:
isin_missing.remove('NSEMCAP Index')

In [160]:
r2_json[:2]

[{'data': [{'figi': 'BBG000DPCRM2',
    'name': 'CIGNITI TECHNOLOGIES LTD',
    'ticker': 'CIGN',
    'exchCode': 'IN',
    'compositeFIGI': 'BBG000DPCRM2',
    'securityType': 'Common Stock',
    'marketSector': 'Equity',
    'shareClassFIGI': 'BBG001SGG9C1',
    'securityType2': 'Common Stock',
    'securityDescription': 'CIGN'}]},
 {'data': [{'figi': 'BBG000CTY013',
    'name': 'KALYANI STEELS LTD',
    'ticker': 'KS',
    'exchCode': 'IN',
    'compositeFIGI': 'BBG000CTY013',
    'securityType': 'Common Stock',
    'marketSector': 'Equity',
    'shareClassFIGI': 'BBG001S83SW9',
    'securityType2': 'Common Stock',
    'securityDescription': 'KS'}]}]

In [200]:
# isin_missing.append("NSEMCAP")
pd.DataFrame(isin_missing).to_clipboard()

In [188]:
payload_dict = [{"idType":"TICKER","idValue":i,"exchCode":"IN"} for i in isin_missing]

In [189]:
payload_dict[:2]

[{'idType': 'TICKER', 'idValue': 'UTNIFTY', 'exchCode': 'IN'},
 {'idType': 'TICKER', 'idValue': 'NHIT', 'exchCode': 'IN'}]

In [192]:
payload = json.dumps(payload_dict)
r3 = requests.post(url, headers=headers, data=payload)

In [193]:
r3_json = r3.json()

In [194]:
l3 = []
for i in range(0,len(r3_json)):
    print(i)
    print(r3_json[i])
    if 'warning' not in r3_json[i]:
        ticker = r3_json[i]['data'][0]['ticker']
    else:
        ticker = np.nan
    l3.append(ticker)

0
{'data': [{'figi': 'BBG009WJZGV5', 'name': 'UTI NIFTY 50 ETF', 'ticker': 'UTNIFTY', 'exchCode': 'IN', 'compositeFIGI': 'BBG009WJZGV5', 'securityType': 'ETP', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG009WJZGW4', 'securityType2': 'Mutual Fund', 'securityDescription': 'UTNIFTY'}]}
1
{'data': [{'figi': 'BBG013BT43X9', 'name': 'NATIONAL HIGHWAYS INFRA TRUS', 'ticker': 'NHIT', 'exchCode': 'IN', 'compositeFIGI': 'BBG013BT43X9', 'securityType': 'Unit', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG013BT43Y8', 'securityType2': 'Unit', 'securityDescription': 'NHIT'}]}
2
{'data': [{'figi': 'BBG000QD5JK3', 'name': 'CENTURY PLYBOARDS INDIA LTD', 'ticker': 'CPBI', 'exchCode': 'IN', 'compositeFIGI': 'BBG000QD5JK3', 'securityType': 'Common Stock', 'marketSector': 'Equity', 'shareClassFIGI': 'BBG001SMN887', 'securityType2': 'Common Stock', 'securityDescription': 'CPBI'}]}
3
{'data': [{'figi': 'BBG00LBY50J0', 'name': "RAINBOW CHILDREN'S MEDICARE", 'ticker': 'RAINBOW', 'exchCode': 'IN', 'compo